In [1]:
# ipca_pipeline.py (or paste into a notebook cell)

from __future__ import annotations
import os
import contextlib
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
from ipca import InstrumentedPCA
%cd /Users/apple/Desktop/Academics/Research_work/TheVirtueOfComplexity_PaperReplication_Experimentation-master
import sys, importlib
import importlib
import src.ipca_workflow as IPCAWorkflow
importlib.reload(IPCAWorkflow)
import ipca
importlib.reload(ipca)
import src.grassmann_ipca_workflow as GrassmannIPCAWorkflow
importlib.reload(GrassmannIPCAWorkflow)


/Users/apple/Desktop/Academics/Research_work/TheVirtueOfComplexity_PaperReplication_Experimentation-master


<module 'src.grassmann_ipca_workflow' from '/Users/apple/Desktop/Academics/Research_work/TheVirtueOfComplexity_PaperReplication_Experimentation-master/src/grassmann_ipca_workflow.py'>

**Download Data from Openap (Run only if you need to download data)**

In [2]:
from src.data_pipeline import DataPipeline

new_mod = DataPipeline()  # or keep your existing instance if already created
char_data = new_mod.download_openap_data(start_yyyymm='199001', end_yyyymm='202512')

KeyboardInterrupt: 

**Generate Data**

In [2]:
from src.IPCA_Grass_estimator import generate_ipca_workflow_panel

char_data, truth = generate_ipca_workflow_panel(
    T=120,                  # months
    N=50,                  # assets
    m=75,                   # characteristics; keep this >= k
    k=18,                   # match your notebook's factor count
    seed=10,
    start_date="1990-01-01",
    include_intercept=False,
)
char_data.head()

,yyyymm,permno,excess_ret,ret_adj,mcap,char_000,char_001,char_002,char_003,char_004,...,char_066,char_067,char_068,char_069,char_070,char_071,char_072,char_073,char_074,y_ipca
0,1990-01-01,10000,0.564907,0.564907,22792.635702,0.354263,-0.065838,-0.088965,-0.081834,-0.071931,...,0.000000,0.558793,-1.358449,-0.497279,1.289567,-0.894716,-0.412525,0.060486,-0.139865,3.047797
1,1990-01-01,10001,2.435053,2.435053,85797.705295,1.329685,-0.471943,-0.055544,-1.250034,-0.219715,...,-1.665469,-2.099675,-0.123121,-0.500843,-0.295286,-0.655342,1.181666,0.098911,0.645068,0.399372
2,1990-01-01,10002,0.468874,0.468874,74960.836684,-1.812264,-1.769878,-0.728883,-1.824737,0.522184,...,0.145865,-0.571294,0.000000,-0.259078,-0.554466,-1.750961,-2.412420,-0.065481,-0.954810,-0.238265
3,1990-01-01,10003,0.608196,0.608196,13222.734308,-0.449103,-0.321676,0.550300,-0.648140,-0.875558,...,0.049090,-0.198872,-0.889634,-1.044275,-0.934324,1.596870,0.000000,0.593299,0.466988,2.228185
4,1990-01-01,10004,-1.995499,-1.995499,16350.773579,-0.818752,0.000000,0.688387,-1.128038,-0.008245,...,-0.073338,-1.730890,-1.559064,-1.481819,-0.807239,-0.796037,1.265732,0.596350,0.000000,-3.191814


**Data Cleaning**

In [3]:
from src.data_pipeline import DataPipeline

new_mod = DataPipeline()  # or keep your existing instance if already created

char_data, dropped_cols = new_mod.remove_mostly_nan_columns(char_data, max_nan_frac=0.5)
print(f"Dropped columns: {dropped_cols}")

Dropped columns: []


In [4]:
char_data, kept_chars, dropped_chars = new_mod.drop_low_std_and_high_corr(
    char_data,
    min_std=1e-5,   # tune as needed
    max_corr=0.75,  # tune as needed
)
char_data = new_mod.fill_remaining_missing(char_data, use_past_only=True)
char_data_copy = char_data.copy()
char_data.head()


,yyyymm,permno,excess_ret,ret_adj,mcap,y_ipca,char_000,char_001,char_002,char_003,...,char_065,char_066,char_067,char_068,char_069,char_070,char_071,char_072,char_073,char_074
0,1990-01-01,10000,0.564907,0.564907,22792.635702,3.047797,0.354263,-0.065838,-0.088965,-0.081834,...,0.630682,0.000000,0.558793,-1.358449,-0.497279,1.289567,-0.894716,-0.412525,0.060486,-0.139865
1,1990-01-01,10001,2.435053,2.435053,85797.705295,0.399372,1.329685,-0.471943,-0.055544,-1.250034,...,0.870551,-1.665469,-2.099675,-0.123121,-0.500843,-0.295286,-0.655342,1.181666,0.098911,0.645068
2,1990-01-01,10002,0.468874,0.468874,74960.836684,-0.238265,-1.812264,-1.769878,-0.728883,-1.824737,...,-2.098309,0.145865,-0.571294,0.000000,-0.259078,-0.554466,-1.750961,-2.412420,-0.065481,-0.954810
3,1990-01-01,10003,0.608196,0.608196,13222.734308,2.228185,-0.449103,-0.321676,0.550300,-0.648140,...,1.239470,0.049090,-0.198872,-0.889634,-1.044275,-0.934324,1.596870,0.000000,0.593299,0.466988
4,1990-01-01,10004,-1.995499,-1.995499,16350.773579,-3.191814,-0.818752,0.000000,0.688387,-1.128038,...,0.980764,-0.073338,-1.730890,-1.559064,-1.481819,-0.807239,-0.796037,1.265732,0.596350,0.000000


**Characteristics of the dataset:**

In [3]:
out_path = "/Users/apple/Desktop/Academics/char_data.parquet"
char_data = pd.read_parquet(out_path, engine="pyarrow")

**Returns Data**

In [4]:
from src.data_pipeline import DataPipeline

new_mod = DataPipeline()  # or keep your existing instance if already created

returns_data = new_mod.download_sp500_returns_wrds(
    start_date="1990-01-01",
    end_date="2026-01-31",
)

WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


**Data Cleaning**

In [5]:
returns_data = returns_data.rename(columns={"excess_ret": "ret_adj"})
char_data = new_mod.merge_openap_with_crsp_returns(char_data, returns_data)

**Main run Different K**

In [6]:
char_data.head()

,permno,yyyymm,AM,AnnouncementReturn,AssetGrowth,BMdec,Beta,BetaFP,BetaLiquidityPS,BidAskSpread,...,hire,Price,Size,STreversal,excess_ret,prc,shrout,cfacpr,cfacshr,mcap
0,10057,1990-01-01,1.999589,0.007559,0.239249,1.034177,1.051798,1.169400,-0.362514,0.009218,...,-0.039773,-2.327278,-4.166398,0.048276,-0.048276,10.25,6291.0,1.0,1.0,64482.75
1,10057,1990-02-01,2.024275,0.007559,0.239249,1.034177,1.064015,1.175841,-0.364891,0.008374,...,-0.039773,-2.315008,-4.154128,0.012195,-0.012195,10.125,6291.0,1.0,1.0,63696.375
2,10057,1990-03-01,2.249233,0.007559,-0.056259,1.102799,1.048140,1.143656,-0.380739,0.011821,...,-0.027473,-2.264364,-4.103484,0.049383,-0.049383,9.625,6291.0,1.0,1.0,60550.875
3,10057,1990-04-01,2.372478,0.000431,-0.056259,1.102799,1.044639,1.178724,-0.384304,0.007117,...,-0.027473,-2.211018,-4.050138,0.041558,-0.041558,9.125,6291.0,1.0,1.0,57405.375
4,10057,1990-05-01,2.405429,0.000431,-0.056259,1.102799,1.033924,1.203003,-0.372249,0.010140,...,-0.027473,-2.197225,-4.036345,0.013699,-0.013699,9.0,6291.0,1.0,1.0,56619.0


**Main Run Different Complexity**

**Runs with multiple settings**

In [16]:
from src.grassmann_ipca_workflow import GrassmannIPCAWorkflow

alphas = [100]
z = 100
wf = GrassmannIPCAWorkflow()
result_rff_all = {}
factors = [18]
k = 18
seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100,110,120]
seed_for_zero = 10  # only used as the storage key for the no-RFF result
T = 36
# initialize result dict
for seed in seeds:
    result_rff_all[seed] = {}

# run the no-RFF case once
if z != 1e-8:
    all_metrics = wf.rolling_ipca_predictions(
        char_data=char_data,
        forecast_start="1993-01-01",
        target_col="y_ipca",
        n_factors=k,
        train_window_months=T,
        min_train_obs=500,
        normalize=True,
        mean_factor=True,
        silent=True,
        iter_tol=1e-4,
        warm_start=True,
        use_rff=False,
        rff_n_components=0,
        rff_gamma=0.25,
        rff_random_state=seed_for_zero,  # irrelevant when use_rff=False
        alpha=z,
        show_progress=True,
        market_cap_filter_col="mcap",
        market_cap_filter_top_n=50,
    )
    result_rff_all[seed_for_zero][0] = all_metrics

seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100,110,120]
rff_n_components = [24,32,48,64,128,256]

for seed in seeds:
    for rff_n_component in rff_n_components:
        all_metrics = wf.rolling_ipca_predictions(
            char_data=char_data,
            forecast_start="1993-01-01",
            target_col="y_ipca",
            n_factors=k,
            train_window_months=T,
            min_train_obs=500,
            normalize=True,
            mean_factor=True,
            silent=True,
            iter_tol=1e-4,
            warm_start=True,
            use_rff=True,
            rff_n_components=rff_n_component,
            rff_gamma=0.25,
            rff_random_state=seed,
            alpha=z,
            show_progress=True,
            market_cap_filter_col="mcap",
            market_cap_filter_top_n=50,
        )
        result_rff_all[seed][rff_n_component] = all_metrics

# run all nonzero RFF configs for every seed
seeds = [10, 20, 30]
rff_n_components = [512, 1024, 2048, 4096]
for seed in seeds:
    for rff_n_component in rff_n_components:
        all_metrics = wf.rolling_ipca_predictions(
            char_data=char_data,
            forecast_start="1993-01-01",
            target_col="y_ipca",
            n_factors=k,
            train_window_months=T,
            min_train_obs=500,
            normalize=True,
            mean_factor=True,
            silent=True,
            iter_tol=1e-4,
            warm_start=True,
            use_rff=True,
            rff_n_components=rff_n_component,
            rff_gamma=0.25,
            rff_random_state=seed,
            alpha=z,
            show_progress=True,
            market_cap_filter_col="mcap",
            market_cap_filter_top_n=50,
        )
        result_rff_all[seed][rff_n_component] = all_metrics


Rolling IPCA:   0%|          | 0/384 [00:00<?, ?month/s]

Rolling IPCA:   0%|          | 0/384 [00:00<?, ?month/s]

Rolling IPCA:   0%|          | 0/384 [00:00<?, ?month/s]

Rolling IPCA:   0%|          | 0/384 [00:00<?, ?month/s]

Rolling IPCA:   0%|          | 0/384 [00:00<?, ?month/s]

...

Rolling IPCA:   0%|          | 0/384 [00:00<?, ?month/s]

Rolling IPCA:   0%|          | 0/384 [00:00<?, ?month/s]

Rolling IPCA:   0%|          | 0/384 [00:00<?, ?month/s]

Rolling IPCA:   0%|          | 0/384 [00:00<?, ?month/s]

Rolling IPCA:   0%|          | 0/384 [00:00<?, ?month/s]

In [17]:
import pickle

with open("my_data_23.pkl", "wb") as f:
    pickle.dump(result_rff_all, f)

**R2calcs**

In [18]:
rff_n_components = [0, 24, 32, 48, 64, 128, 256, 512, 1024, 2048, 4096]
oos_r2_results = dict()  # or a list if you prefer
for rff_n_component in rff_n_components:
    oos_r2_results[rff_n_component] = []
oos_r2_results[0] = []  # for the no-RFF case  
for factor, metrics in result_rff_all.items():
    for rff_n_component, all_metrics in metrics.items():
        pred_df_in = all_metrics[0]
        ss_res = ((pred_df_in["y_true"] - pred_df_in["y_pred"]) ** 2).sum()
        ss_tot = (pred_df_in["y_true"] ** 2).sum()
        #print(f"Factor: {factor}, RFF n_components: {rff_n_component}, SS_res: {ss_res:.4f}, SS_tot: {ss_tot:.4f}")
        r2_oos = 1 - ss_res / ss_tot
        oos_r2_results[rff_n_component].append(r2_oos)
print("OOS R² results by RFF component count:")
for rff_n_component, r2_values in oos_r2_results.items():
    avg_r2 = np.mean(r2_values)
    print(f"RFF n_components={rff_n_component}: Average OOS R² = {avg_r2:.4f} over {len(r2_values)} runs")

OOS R² results by RFF component count:
RFF n_components=0: Average OOS R² = -0.0326 over 1 runs
RFF n_components=24: Average OOS R² = 0.0040 over 12 runs
RFF n_components=32: Average OOS R² = 0.0041 over 12 runs
RFF n_components=48: Average OOS R² = 0.0052 over 12 runs
RFF n_components=64: Average OOS R² = 0.0050 over 12 runs
RFF n_components=128: Average OOS R² = 0.0056 over 12 runs
RFF n_components=256: Average OOS R² = 0.0064 over 12 runs
RFF n_components=512: Average OOS R² = 0.0071 over 3 runs
RFF n_components=1024: Average OOS R² = 0.0084 over 3 runs
RFF n_components=2048: Average OOS R² = 0.0081 over 3 runs
RFF n_components=4096: Average OOS R² = 0.0088 over 3 runs


In [19]:
features_data = {}
for seeds, metrics in result_rff_all.items():
    for rff_n_component, all_metrics in metrics.items():
        pred_df_in = all_metrics[1]
        features_data[(seeds, rff_n_component)] = pred_df_in.describe(include="all").T.reset_index().rename(columns={"index": "column"})

In [20]:
features_data_df = pd.concat(features_data.values(), keys=features_data.keys()).reset_index(level=[0,1]).rename(columns={"level_0": "seeds", "level_1": "rff_n_component"})
# Convert summary-stat columns to numeric before aggregation (object -> float)
# Safe groupby mean on numeric columns only
num_cols = [c for c in ["count", "mean", "min", "25%", "50%", "75%", "max", "std"] if c in features_data_df.columns]
features_data_df[num_cols] = features_data_df[num_cols].apply(pd.to_numeric, errors="coerce")

features_data_df = (
    features_data_df
    .groupby(["rff_n_component", "column"], as_index=False)[num_cols]
    .mean()
)
features_data_df.to_excel("features_data_18_months_36.xlsx", index=False)
features_data_df

,rff_n_component,column,count,mean,min,25%,50%,75%,max,std
0,0,d_proj,383.0,2.056487,0.546272,1.793531,2.082463,2.339168,3.130814,0.447350
1,0,d_proj_norm,383.0,0.408751,0.108578,0.356485,0.413914,0.464937,0.622286,0.088916
2,0,erank,384.0,12.776471,10.796903,11.934308,12.660454,13.586330,14.934395,0.997586
3,0,gap_ratio,384.0,0.059289,0.030797,0.045614,0.053329,0.070564,0.113901,0.018397
4,0,geodesic_acceleration,382.0,0.001909,-1.583712,-0.344703,-0.014243,0.394498,1.383347,0.554531
...,...,...,...,...,...,...,...,...,...,...
116,4096,n_test,384.0,49.953125,48.000000,50.000000,50.000000,50.000000,50.000000,0.223643
117,4096,n_train,384.0,1772.299479,1685.000000,1760.000000,1774.000000,1788.000000,1800.000000,21.436320
118,4096,principal_angle_max,383.0,0.738812,0.492532,0.653514,0.727781,0.809275,1.361569,0.120337
119,4096,principal_angle_mean,383.0,0.247163,0.059845,0.176331,0.237250,0.309652,0.532136,0.095545


In [21]:
import pandas as pd
import importlib
import src.portfolio_utils as portfolio_utils
importlib.reload(portfolio_utils)

from src.portfolio_utils import (
    build_directional_portfolio,
    compute_portfolio_returns,
    portfolio_performance,
)

mean_prediction_by_rff = {}
portfolio_results_mean_seed = {}

rff_components = sorted({
    rff_n_component
    for seed_metrics in result_rff_all.values()
    for rff_n_component in seed_metrics
})

for rff_n_component in rff_components:
    seed_pred_list = []

    for seed, seed_metrics in result_rff_all.items():
        if rff_n_component not in seed_metrics:
            continue

        pred_df = seed_metrics[rff_n_component][0].copy()
        pred_df["yyyymm"] = pd.to_datetime(pred_df["yyyymm"])
        pred_df["seed"] = seed

        seed_pred_list.append(
            pred_df[
                ["permno", "yyyymm", "y_true", "y_pred", "train_end", "n_train", "n_test", "seed"]
            ]
        )

    if not seed_pred_list:
        continue

    stacked = pd.concat(seed_pred_list, ignore_index=True)

    # sanity check: realized return should be the same across seeds
    y_true_check = stacked.groupby(["permno", "yyyymm"])["y_true"].nunique()
    if (y_true_check > 1).any():
        raise ValueError(
            f"y_true is not identical across seeds for rff_n_component={rff_n_component}"
        )

    mean_pred_df = (
        stacked
        .groupby(["permno", "yyyymm"], as_index=False)
        .agg(
            y_true=("y_true", "first"),
            y_pred=("y_pred", "mean"),
            train_end=("train_end", "first"),
            n_train=("n_train", "first"),
            n_test=("n_test", "first"),
            n_seeds=("seed", "nunique"),
        )
        .sort_values(["yyyymm", "permno"])
        .reset_index(drop=True)
    )

    mean_prediction_by_rff[rff_n_component] = mean_pred_df

    port = build_directional_portfolio(mean_pred_df)
    monthly_ret = compute_portfolio_returns(port)
    stats = portfolio_performance(monthly_ret)

    portfolio_results_mean_seed[rff_n_component] = {
        "mean_pred_df": mean_pred_df,
        "portfolio": port,
        "returns": monthly_ret,
        "performance": stats,
    }

perf_table = pd.DataFrame(
    {k: v["performance"] for k, v in portfolio_results_mean_seed.items()}
).T.sort_index()

perf_table


Annualized Return : 10.18%
Annualized Vol    : 15.69%
Sharpe Ratio      : 0.65
Max Drawdown      : -46.55%
Hit Rate          : 61.88%
Annualized Return : 8.74%
Annualized Vol    : 10.41%
Sharpe Ratio      : 0.84
Max Drawdown      : -28.61%
Hit Rate          : 63.71%
Annualized Return : 8.64%
Annualized Vol    : 10.48%
Sharpe Ratio      : 0.82
Max Drawdown      : -24.32%
Hit Rate          : 62.14%
Annualized Return : 9.86%
Annualized Vol    : 10.64%
Sharpe Ratio      : 0.93
Max Drawdown      : -23.87%
Hit Rate          : 64.23%
Annualized Return : 8.48%
Annualized Vol    : 10.82%
Sharpe Ratio      : 0.78
Max Drawdown      : -27.75%
Hit Rate          : 63.19%
Annualized Return : 8.59%
Annualized Vol    : 11.18%
Sharpe Ratio      : 0.77
Max Drawdown      : -37.80%
Hit Rate          : 64.75%
Annualized Return : 8.41%
Annualized Vol    : 11.27%
Sharpe Ratio      : 0.75
Max Drawdown      : -32.00%
Hit Rate          : 61.88%
Annualized Return : 8.44%
Annualized Vol    : 10.91%
Sharpe Ratio   

,ann_ret,ann_vol,sharpe,max_dd,hit_rate
0,0.101783,0.156928,0.648599,-0.465452,0.618799
24,0.087443,0.104084,0.840120,-0.286062,0.637076
32,0.086387,0.104755,0.824659,-0.243231,0.621410
48,0.098611,0.106410,0.926707,-0.238652,0.642298
64,0.084842,0.108236,0.783858,-0.277477,0.631854
128,0.085903,0.111838,0.768098,-0.377970,0.647520
256,0.084118,0.112687,0.746470,-0.320009,0.618799
512,0.084416,0.109150,0.773397,-0.324585,0.634465
1024,0.096346,0.110258,0.873825,-0.265651,0.650131
2048,0.083280,0.116804,0.712988,-0.352296,0.629243


In [22]:
perf_table.to_excel("portfolio_performance_18_months_36.xlsx", index=True)

In [23]:
perf_table

,ann_ret,ann_vol,sharpe,max_dd,hit_rate
0,0.101783,0.156928,0.648599,-0.465452,0.618799
24,0.087443,0.104084,0.840120,-0.286062,0.637076
32,0.086387,0.104755,0.824659,-0.243231,0.621410
48,0.098611,0.106410,0.926707,-0.238652,0.642298
64,0.084842,0.108236,0.783858,-0.277477,0.631854
128,0.085903,0.111838,0.768098,-0.377970,0.647520
256,0.084118,0.112687,0.746470,-0.320009,0.618799
512,0.084416,0.109150,0.773397,-0.324585,0.634465
1024,0.096346,0.110258,0.873825,-0.265651,0.650131
2048,0.083280,0.116804,0.712988,-0.352296,0.629243


In [13]:
portfolio_performance.to_excel("portfolio_performance_factors_seeds_14.xlsx", index=False)

In [44]:
features_data_df

,rff_n_component,column,count,mean,min,25%,50%,75%,max,std
0,0,d_proj,383.0,1.911702,2.980232e-08,1.724219,1.972377,2.193887,2.875697,0.438990
1,0,d_proj_norm,383.0,0.437139,6.814741e-09,0.394268,0.451013,0.501665,0.657571,0.100381
2,0,erank,384.0,9.157032,7.874873e+00,8.624270,9.088635,9.669800,10.693293,0.661438
3,0,gap_ratio,384.0,0.097937,3.557939e-02,0.073640,0.090362,0.115632,0.222455,0.031827
4,0,geodesic_acceleration,382.0,0.002153,-2.405906e+00,-0.270716,-0.003919,0.291729,2.339773,0.574877
...,...,...,...,...,...,...,...,...,...,...
127,512,n_test,384.0,49.953125,4.800000e+01,50.000000,50.000000,50.000000,50.000000,0.223643
128,512,n_train,384.0,1188.156250,1.133000e+03,1181.000000,1190.000000,1200.000000,1200.000000,12.042855
129,512,principal_angle_max,383.0,1.077263,6.535935e-01,0.954442,1.067447,1.192536,1.561510,0.174164
130,512,principal_angle_mean,383.0,0.425663,1.666950e-01,0.353302,0.426240,0.496762,0.752757,0.104719


In [18]:
char_data.columns

Index(['permno', 'yyyymm', 'AM', 'AnnouncementReturn', 'AssetGrowth', 'BMdec',
       'Beta', 'BetaFP', 'BetaLiquidityPS', 'BidAskSpread', 'BookLeverage',
       'CF', 'ChEQ', 'ChInv', 'ChNWC', 'ConvDebt', 'CoskewACX', 'Coskewness',
       'CredRatDG', 'DebtIssuance', 'DelCOA', 'DelLTI', 'DivInit', 'DivOmit',
       'EquityDuration', 'ExchSwitch', 'FirmAge', 'Herf', 'High52',
       'IdioVolAHT', 'Illiquidity', 'IndIPO', 'IndMom', 'IntMom', 'LRreversal',
       'MRreversal', 'MaxRet', 'Mom12m', 'Mom12mOffSeason', 'Mom6m',
       'MomOffSeason', 'MomSeason', 'MomSeasonShort', 'NetDebtFinance',
       'NetEquityFinance', 'NumEarnIncrease', 'PctAcc', 'PriceDelayRsq',
       'PriceDelaySlope', 'PriceDelayTstat', 'RDIPO', 'RDS',
       'ResidualMomentum', 'ReturnSkew3F', 'SP', 'ShareRepurchase', 'Spinoff',
       'Tax', 'VolSD', 'VolumeTrend', 'betaVIX', 'hire', 'Price', 'Size',
       'STreversal', 'excess_ret', 'prc', 'shrout', 'cfacpr', 'cfacshr',
       'mcap'],
      dtype='object')